# Phase 3 — Perturbation Engine
**Student ID: 5720570 | University of Warwick**

Builds and validates the three meaning-preserving perturbation types
from dissertation specification Table 1.

- Perturbation 1: Lexical Formality (informal to formal connectives)
- Perturbation 2: Connective Density (add non-functional transitions)  
- Perturbation 3: Syntactic Voice (active to passive)

Each perturbation is verified through the SBERT semantic gate (cosine similarity >= 0.90)

In [36]:
import pandas as pd
import numpy as np
import re
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")
print(f"spaCy version  : {spacy.__version__}")

Libraries imported
spaCy version  : 3.8.11


In [37]:
# Run this once to install the spaCy English model
import subprocess
result = subprocess.run(
    ['python', '-m', 'spacy', 'download', 'en_core_web_sm'],
    capture_output=True, text=True
)
print(result.stdout[-200:] if result.stdout else "Already installed")

import en_core_web_sm
nlp = en_core_web_sm.load()
print("spaCy en_core_web_sm loaded")

 you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

spaCy en_core_web_sm loaded


## 1. Load Essays
Load the train split essays — these are the 12,464 essays used for perturbation testing.

In [38]:
DATA_PATH = '/kaggle/input/datasets/shravaniuk/persuade-corpus-2-0-tt'

df_raw = pd.read_csv(f'{DATA_PATH}/persuade_corpus_2.0_train.csv')

KEEP_COLS = ['essay_id', 'full_text', 'holistic_essay_score', 
             'prompt_name', 'grade_level', 'ell_status',
             'student_disability_status', 'economically_disadvantaged',
             'gender', 'race_ethnicity']

df = (df_raw[KEEP_COLS]
        .drop_duplicates(subset='essay_id')
        .reset_index(drop=True))

df = df.rename(columns={'holistic_essay_score': 'score'})

print(f"Essays loaded    : {len(df):,}")
print(f"Score range      : {df['score'].min()} to {df['score'].max()}")
print(f"Prompts          : {df['prompt_name'].nunique()}")
print(f"Null full_text   : {df['full_text'].isna().sum()}")

Essays loaded    : 15,593
Score range      : 1 to 6
Prompts          : 15
Null full_text   : 0


## 2. Perturbation 1 — Lexical Formality
Replaces informal connective markers with formal equivalents using regex.
Rule-based and deterministic — no neural model involved.
Rationale: tests if BERT rewards formal register over logical content (spec Table 1, Row 1).

In [39]:
# Ordered from longest to shortest to prevent partial replacements
# e.g. "But also" must be matched before "But"
FORMALITY_RULES = [
    (r'\bBut also\b',       'Moreover'),
    (r'\bbut also\b',       'moreover'),
    (r'\bBut\b',            'However'),
    (r'\bbut\b',            'however'),
    (r'\bAlso\b',           'Furthermore'),
    (r'\balso\b',           'furthermore'),
    (r'\bSo\b',             'Therefore'),
    (r'\bso\b',             'therefore'),
    (r'\bPlus\b',           'In addition'),
    (r'\bplus\b',           'in addition'),
    (r'\bAnd\b',            'Additionally'),
    (r'(?<=[.!?,])\s+[Aa]nd\b', ' Additionally'),
    (r'\bThough\b',         'Nevertheless'),
    (r'\bthough\b',         'nevertheless'),
    (r'\bYet\b',            'Nonetheless'),
    (r'\byet\b',            'nonetheless'),
]

def perturb_lexical_formality(text):
    """
    Apply informal to formal connective swaps.
    Returns perturbed text and count of substitutions made.
    """
    perturbed = str(text)
    total_subs = 0
    for pattern, replacement in FORMALITY_RULES:
        new_text, n = re.subn(pattern, replacement, perturbed)
        perturbed = new_text
        total_subs += n
    return perturbed, total_subs


# Test on sample sentences
test_cases = [
    "The policy is wrong. But many people support it. So we need to think carefully.",
    "Climate change is real. And it is getting worse. Also, we are running out of time.",
    "I agree with the author. But also I think there are other issues to consider.",
    "Students study hard. Yet they sometimes fail. Though this is not always fair.",
]

print("Lexical Formality — Test Results")
print("=" * 60)
for text in test_cases:
    result, n_subs = perturb_lexical_formality(text)
    print(f"ORIGINAL  : {text}")
    print(f"PERTURBED : {result}")
    print(f"Subs made : {n_subs}")
    print("-" * 60)

Lexical Formality — Test Results
ORIGINAL  : The policy is wrong. But many people support it. So we need to think carefully.
PERTURBED : The policy is wrong. However many people support it. Therefore we need to think carefully.
Subs made : 2
------------------------------------------------------------
ORIGINAL  : Climate change is real. And it is getting worse. Also, we are running out of time.
PERTURBED : Climate change is real. Additionally it is getting worse. Furthermore, we are running out of time.
Subs made : 2
------------------------------------------------------------
ORIGINAL  : I agree with the author. But also I think there are other issues to consider.
PERTURBED : I agree with the author. Moreover I think there are other issues to consider.
Subs made : 1
------------------------------------------------------------
ORIGINAL  : Students study hard. Yet they sometimes fail. Though this is not always fair.
PERTURBED : Students study hard. Nonetheless they sometimes fail. Never

## 3. Perturbation 2 — Connective Density
Inserts non-functional academic transition phrases at the beginning of sentences.
Rotates through a list to avoid repetition.
Rationale: tests if BERT uses transition-word frequency as proxy for organisation (spec Table 1, Row 3).

In [40]:
DENSITY_CONNECTIVES = [
    'Furthermore, ',
    'Additionally, ',
    'Moreover, ',
    'In this regard, ',
    'It is worth noting that ',
    'Significantly, ',
    'Indeed, ',
    'Consequently, ',
]

def perturb_connective_density(text):
    """
    Insert non-functional transition phrases at sentence starts.
    Uses rotation to vary the inserted phrases.
    Skips very short sentences (< 5 words) and sentences 
    that already start with a connective.
    Returns perturbed text and count of insertions.
    """
    # Split into sentences using simple boundary detection
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    
    # Connectives that indicate sentence already has a transition
    existing_connectives = {
        'however', 'furthermore', 'moreover', 'additionally',
        'therefore', 'nevertheless', 'nonetheless', 'consequently',
        'in addition', 'in conclusion', 'in summary', 'finally',
        'firstly', 'secondly', 'thirdly', 'lastly', 'indeed',
        'significantly', 'notably', 'importantly', 'thus'
        'in addition', 'in conclusion', 'in summary', 'in contrast',
        'in particular', 'in fact', 'in other words', 'in short'
    }
    
    new_sentences = []
    insert_count  = 0
    conn_index    = 0
    
    for i, sent in enumerate(sentences):
        words = sent.strip().split()
        first_word = words[0].lower().rstrip(',') if words else ''
        
        # Skip: first sentence, short sentences, already has connective
        if i == 0:
            new_sentences.append(sent)
            continue
        if len(words) < 5:
            new_sentences.append(sent)
            continue
        if first_word in existing_connectives:
            new_sentences.append(sent)
            continue
            
        # Insert connective
        connective = DENSITY_CONNECTIVES[conn_index % len(DENSITY_CONNECTIVES)]
        # Lowercase first letter of original sentence after insertion
        if sent and sent[0].isupper():
            modified = connective + sent[0].lower() + sent[1:]
        else:
            modified = connective + sent
        new_sentences.append(modified)
        conn_index   += 1
        insert_count += 1
    
    return ' '.join(new_sentences), insert_count


# Test
test_cases_2 = [
    "Climate change is a serious problem. We must act now. The evidence is clear. Scientists agree on this.",
    "Education is important. Students deserve quality teaching. Schools need more funding. Teachers are overworked.",
]

print("Connective Density — Test Results")
print("=" * 60)
for text in test_cases_2:
    result, n_ins = perturb_connective_density(text)
    print(f"ORIGINAL  : {text}")
    print(f"PERTURBED : {result}")
    print(f"Insertions: {n_ins}")
    print("-" * 60)

Connective Density — Test Results
ORIGINAL  : Climate change is a serious problem. We must act now. The evidence is clear. Scientists agree on this.
PERTURBED : Climate change is a serious problem. We must act now. The evidence is clear. Scientists agree on this.
Insertions: 0
------------------------------------------------------------
ORIGINAL  : Education is important. Students deserve quality teaching. Schools need more funding. Teachers are overworked.
PERTURBED : Education is important. Students deserve quality teaching. Schools need more funding. Teachers are overworked.
Insertions: 0
------------------------------------------------------------


## 4. Perturbation 3 — Syntactic Voice (Active to Passive)
Converts active voice sentences to passive voice using spaCy dependency parsing.
Targets simple SVO (Subject-Verb-Object) sentences only.
Rationale: tests sensitivity to structural complexity unrelated to argumentative strength (spec Table 1, Row 2).

In [41]:
# Irregular verb past participle lookup
IRREGULAR_PAST_PARTICIPLE = {
    'be': 'been', 'have': 'had', 'do': 'done', 'say': 'said',
    'go': 'gone', 'get': 'gotten', 'make': 'made', 'know': 'known',
    'think': 'thought', 'take': 'taken', 'see': 'seen', 'come': 'come',
    'want': 'wanted', 'look': 'looked', 'use': 'used', 'find': 'found',
    'give': 'given', 'tell': 'told', 'work': 'worked', 'call': 'called',
    'try': 'tried', 'ask': 'asked', 'need': 'needed', 'feel': 'felt',
    'become': 'become', 'leave': 'left', 'put': 'put', 'mean': 'meant',
    'keep': 'kept', 'let': 'let', 'begin': 'begun', 'show': 'shown',
    'hear': 'heard', 'play': 'played', 'run': 'run', 'move': 'moved',
    'live': 'lived', 'believe': 'believed', 'hold': 'held', 'bring': 'brought',
    'happen': 'happened', 'write': 'written', 'provide': 'provided',
    'sit': 'sat', 'stand': 'stood', 'lose': 'lost', 'pay': 'paid',
    'meet': 'met', 'include': 'included', 'continue': 'continued',
    'set': 'set', 'learn': 'learned', 'change': 'changed', 'lead': 'led',
    'understand': 'understood', 'watch': 'watched', 'follow': 'followed',
    'create': 'created', 'affect': 'affected', 'cause': 'caused',
    'drive': 'driven', 'build': 'built', 'spend': 'spent', 'cut': 'cut',
    'send': 'sent', 'read': 'read', 'grow': 'grown', 'buy': 'bought',
    'develop': 'developed', 'increase': 'increased', 'reduce': 'reduced',
    'support': 'supported', 'argue': 'argued', 'suggest': 'suggested',
    'allow': 'allowed', 'help': 'helped', 'start': 'started',
    'show': 'shown', 'consider': 'considered', 'ban': 'banned',
    'introduce': 'introduced', 'assess': 'assessed', 'achieve': 'achieved',
    'protect': 'protected', 'require': 'required', 'force': 'forced',
    'encourage': 'encouraged', 'prevent': 'prevented', 'destroy': 'destroyed',
    'save': 'saved', 'kill': 'killed', 'hurt': 'hurt', 'harm': 'harmed',
    'risk': 'risked', 'cause': 'caused', 'improve': 'improved',
}

def get_past_participle(verb_lemma):
    """Get past participle — use lookup or add 'ed' for regular verbs."""
    if verb_lemma in IRREGULAR_PAST_PARTICIPLE:
        return IRREGULAR_PAST_PARTICIPLE[verb_lemma]
    # Regular verb: ends in e → just add d, otherwise add ed
    if verb_lemma.endswith('e'):
        return verb_lemma + 'd'
    if verb_lemma.endswith('y') and len(verb_lemma) > 2:
        return verb_lemma[:-1] + 'ied'
    return verb_lemma + 'ed'


def convert_to_passive(sentence_text, nlp_model):
    """
    Converts a simple active SVO sentence to passive voice.
    Only applies when clear subject, transitive verb, and object exist.
    """
    doc = nlp_model(sentence_text.strip())

    subject = None
    verb    = None
    obj     = None

    for token in doc:
        if token.dep_ == 'nsubj' and token.head.pos_ == 'VERB':
            subject = token
            verb    = token.head
        if token.dep_ == 'dobj' and token.head == verb:
            obj = token

    if not (subject and verb and obj):
        return sentence_text, False

    # Safety checks — skip pronouns as subject (produces "is done by I/he/they")
    if subject.pos_ == 'PRON':
        return sentence_text, False

    # Skip if object is a pronoun
    if obj.pos_ == 'PRON':
        return sentence_text, False

    subj_span = doc[subject.left_edge.i : subject.right_edge.i + 1]
    obj_span  = doc[obj.left_edge.i     : obj.right_edge.i + 1]

    past_participle = get_past_participle(verb.lemma_)

    # Determine correct to-be verb based on object number
    if obj.tag_ in ('NNS', 'NNPS') or obj.dep_ == 'nsubj':
        to_be = 'are'
    else:
        to_be = 'is'

    # Build passive sentence — lowercase first letter of original subject
    obj_text  = obj_span.text
    subj_text = subj_span.text[0].lower() + subj_span.text[1:] if subj_span.text else ''

    passive = f"{obj_text} {to_be} {past_participle} by {subj_text}"
    passive = passive[0].upper() + passive[1:]

    return passive + '.', True


def perturb_syntactic_voice(text, nlp_model):
    """
    Apply active-to-passive conversion on eligible sentences.
    Returns perturbed text and count of conversions.
    """
    sentences     = re.split(r'(?<=[.!?])\s+', str(text).strip())
    new_sents     = []
    convert_count = 0

    for sent in sentences:
        words = sent.strip().split()
        # Skip very short sentences
        if len(words) < 4:
            new_sents.append(sent)
            continue
        converted, success = convert_to_passive(sent, nlp_model)
        if success:
            new_sents.append(converted)
            convert_count += 1
        else:
            new_sents.append(sent)

    return ' '.join(new_sents), convert_count


# Test with same examples
test_cases_3 = [
    "The student writes the essay carefully.",
    "Teachers assess student work every semester.",
    "The government introduced new education policies.",
    "Climate change affects millions of people worldwide.",
    "Experts support the new educational framework.",
    "Students complete assignments every week.",
]

print("Syntactic Voice — Fixed Results")
print("=" * 60)
for text in test_cases_3:
    result, n_conv = perturb_syntactic_voice(text, nlp)
    print(f"ORIGINAL  : {text}")
    print(f"PERTURBED : {result}")
    print(f"Converted : {n_conv} sentences")
    print("-" * 60)

Syntactic Voice — Fixed Results
ORIGINAL  : The student writes the essay carefully.
PERTURBED : The essay is written by the student.
Converted : 1 sentences
------------------------------------------------------------
ORIGINAL  : Teachers assess student work every semester.
PERTURBED : Student work is assessed by teachers.
Converted : 1 sentences
------------------------------------------------------------
ORIGINAL  : The government introduced new education policies.
PERTURBED : New education policies are introduced by the government.
Converted : 1 sentences
------------------------------------------------------------
ORIGINAL  : Climate change affects millions of people worldwide.
PERTURBED : Millions of people are affected by climate change.
Converted : 1 sentences
------------------------------------------------------------
ORIGINAL  : Experts support the new educational framework.
PERTURBED : The new educational framework is supported by experts.
Converted : 1 sentences
-----------

In [42]:
# Test all 3 perturbations on one real essay from the dataset
sample_essay = df['full_text'].iloc[0]
sample_id    = df['essay_id'].iloc[0]
sample_score = df['score'].iloc[0]

print(f"Essay ID    : {sample_id}")
print(f"True score  : {sample_score}")
print(f"Word count  : {len(sample_essay.split())}")
print()
print("ORIGINAL (first 300 chars):")
print(sample_essay[:300])
print()

p1, n1 = perturb_lexical_formality(sample_essay)
p2, n2 = perturb_connective_density(sample_essay)
p3, n3 = perturb_syntactic_voice(sample_essay, nlp)

print(f"Perturbation 1 (Lexical Formality)   : {n1} substitutions")
print(f"Perturbation 2 (Connective Density)  : {n2} insertions")
print(f"Perturbation 3 (Syntactic Voice)     : {n3} conversions")
print()
print("P1 perturbed (first 300 chars):")
print(p1[:300])
print()
print("P2 perturbed (first 300 chars):")
print(p2[:300])

Essay ID    : 5.40889E+12
True score  : 3
Word count  : 379

ORIGINAL (first 300 chars):
Phones

Modern humans today are always on their phone. They are always on their phone more than 5 hours a day no stop .All they do is text back and forward and just have group Chats on social media. They even do it while driving. They are some really bad consequences when stuff happens when it comes

Perturbation 1 (Lexical Formality)   : 4 substitutions
Perturbation 2 (Connective Density)  : 26 insertions
Perturbation 3 (Syntactic Voice)     : 2 conversions

P1 perturbed (first 300 chars):
Phones

Modern humans today are always on their phone. They are always on their phone more than 5 hours a day no stop .All they do is text back and forward and just have group Chats on social media. They even do it while driving. They are some really bad consequences when stuff happens when it comes

P2 perturbed (first 300 chars):
Phones

Modern humans today are always on their phone. Furthermore, they are alwa

In [43]:
# Check how many essays are affected by each perturbation type
# Run on first 100 essays as a quick coverage audit

sample_100 = df.head(100)

p1_hits = 0
p2_hits = 0
p3_hits = 0

p1_avg_subs  = []
p2_avg_ins   = []
p3_avg_conv  = []

for _, row in sample_100.iterrows():
    text = row['full_text']
    
    _, n1 = perturb_lexical_formality(text)
    _, n2 = perturb_connective_density(text)
    _, n3 = perturb_syntactic_voice(text, nlp)
    
    if n1 > 0: p1_hits += 1; p1_avg_subs.append(n1)
    if n2 > 0: p2_hits += 1; p2_avg_ins.append(n2)
    if n3 > 0: p3_hits += 1; p3_avg_conv.append(n3)

print("=" * 55)
print("PERTURBATION COVERAGE — 100 essay sample")
print("=" * 55)
print(f"P1 Lexical Formality  : {p1_hits}/100 essays affected")
print(f"   Avg substitutions  : {np.mean(p1_avg_subs):.1f} per essay")
print()
print(f"P2 Connective Density : {p2_hits}/100 essays affected")
print(f"   Avg insertions     : {np.mean(p2_avg_ins):.1f} per essay")
print()
print(f"P3 Syntactic Voice    : {p3_hits}/100 essays affected")
print(f"   Avg conversions    : {np.mean(p3_avg_conv):.1f} per essay")
print()
print("Note: P1 and P2 should affect 90+ percent of essays")
print("P3 will have lower coverage as not all sentences are simple SVO")

PERTURBATION COVERAGE — 100 essay sample
P1 Lexical Formality  : 98/100 essays affected
   Avg substitutions  : 5.9 per essay

P2 Connective Density : 100/100 essays affected
   Avg insertions     : 20.3 per essay

P3 Syntactic Voice    : 87/100 essays affected
   Avg conversions    : 3.0 per essay

Note: P1 and P2 should affect 90+ percent of essays
P3 will have lower coverage as not all sentences are simple SVO


In [44]:
# Save first 50 essays with all 3 perturbations
# This becomes the test set for SBERT gate validation in Step 2

records = []
for _, row in df.head(50).iterrows():
    text  = row['full_text']
    eid   = row['essay_id']
    score = row['score']
    
    p1, n1 = perturb_lexical_formality(text)
    p2, n2 = perturb_connective_density(text)
    p3, n3 = perturb_syntactic_voice(text, nlp)
    
    records.append({
        'essay_id'      : eid,
        'score'         : score,
        'original'      : text,
        'p1_formality'  : p1,
        'p1_subs'       : n1,
        'p2_density'    : p2,
        'p2_insertions' : n2,
        'p3_voice'      : p3,
        'p3_conversions': n3,
    })

df_sample = pd.DataFrame(records)
df_sample.to_csv('perturbation_sample_50.csv', index=False)

print(f"Saved: perturbation_sample_50.csv")
print(f"Rows : {len(df_sample)}")
print(f"Cols : {list(df_sample.columns)}")
print()
print("Step 1 complete — ready for Step 2 SBERT validation")

Saved: perturbation_sample_50.csv
Rows : 50
Cols : ['essay_id', 'score', 'original', 'p1_formality', 'p1_subs', 'p2_density', 'p2_insertions', 'p3_voice', 'p3_conversions']

Step 1 complete — ready for Step 2 SBERT validation


In [45]:
# ── P4: DISCOURSE WEAKENING ──────────────────────────────────────────────

def perturb_discourse_weakening(essay_id, df_discourse):
    """
    Find an Effective segment in the essay.
    Replace it with an Ineffective segment of the same discourse_type
    from another essay. Return perturbed full text.
    """
    essay_rows = df_discourse[df_discourse['essay_id'] == essay_id].copy()
    full_text   = essay_rows['full_text'].iloc[0]

    # Find Effective segments in this essay
    effective_segs = essay_rows[essay_rows['discourse_effectiveness'] == 'Effective']
    if len(effective_segs) == 0:
        return full_text, False  # No effective segment to weaken

    # Pick first effective segment
    target = effective_segs.iloc[0]
    disc_type = target['discourse_type']

    # Find an Ineffective segment of same type from a DIFFERENT essay
    replacements = df_discourse[
        (df_discourse['discourse_effectiveness'] == 'Ineffective') &
        (df_discourse['discourse_type'] == disc_type) &
        (df_discourse['essay_id'] != essay_id)
    ]
    if len(replacements) == 0:
        return full_text, False  # No suitable replacement found

    replacement_text = replacements.sample(1, random_state=42)['discourse_text'].iloc[0]
    original_text    = target['discourse_text']

    # Replace in full text (first occurrence only)
    if original_text in full_text:
        perturbed = full_text.replace(original_text, replacement_text, 1)
        return perturbed, True

    return full_text, False


print("P4 Discourse Weakening function defined")
print("This uses discourse_effectiveness annotations from PERSUADE 2.0")
print("Replaces one Effective segment with Ineffective of same discourse_type")

P4 Discourse Weakening function defined
This uses discourse_effectiveness annotations from PERSUADE 2.0
Replaces one Effective segment with Ineffective of same discourse_type


In [46]:
# ── P5: STRUCTURAL DISRUPTION ────────────────────────────────────────────
# Removes the Counterclaim or Rebuttal discourse segment from essays
# that contain one. Tests whether BERT detects loss of argument structure.

def perturb_structural_disruption(essay_id, df_discourse):
    """
    Remove the Counterclaim or Rebuttal segment from the essay.
    Returns perturbed text and whether disruption was applied.
    """
    essay_rows = df_discourse[df_discourse['essay_id'] == essay_id].copy()
    full_text  = essay_rows['full_text'].iloc[0]

    # Find Counterclaim or Rebuttal segments
    structural_segs = essay_rows[
        essay_rows['discourse_type'].isin(['Counterclaim', 'Rebuttal'])
    ]
    if len(structural_segs) == 0:
        return full_text, False  # Essay has no counterclaim/rebuttal

    # Remove the first structural segment found
    target_text = structural_segs.iloc[0]['discourse_text']

    if target_text in full_text:
        perturbed = full_text.replace(target_text, '', 1)
        # Clean up extra whitespace
        import re
        perturbed = re.sub(r'\n\s*\n', '\n', perturbed).strip()
        return perturbed, True

    return full_text, False


print("P5 Structural Disruption function defined")
print("Removes Counterclaim or Rebuttal segment from essays that contain one")
print("Tests whether BERT detects loss of argumentative structure")

P5 Structural Disruption function defined
Removes Counterclaim or Rebuttal segment from essays that contain one
Tests whether BERT detects loss of argumentative structure


In [47]:
# ── P6: LENGTH EXTENSION ─────────────────────────────────────────────────
# need counterfactual evidence before claiming length causes
# score differences. This adds filler sentences that repeat existing points
# without adding new arguments. If score rises, length alone drives scoring.

FILLER_TEMPLATES = [
    "This point has been discussed above and remains an important consideration.",
    "As mentioned previously, this aspect of the argument deserves attention.",
    "The ideas presented earlier in this essay continue to support this view.",
    "This further reinforces the argument that has been made throughout this essay.",
    "It is therefore clear, based on the points raised above, that this is significant.",
]

def perturb_length_extension(text, n_fillers=3):
    """
    Appends n_filler filler sentences to the end of the essay.
    These sentences add no new argument — only increase word count.
    Returns perturbed text and number of sentences added.
    """
    import random
    fillers = random.sample(FILLER_TEMPLATES, min(n_fillers, len(FILLER_TEMPLATES)))
    filler_block = ' '.join(fillers)
    perturbed = str(text).strip() + ' ' + filler_block
    return perturbed, len(fillers)


# Test
test_text = "Climate change is a serious issue. We must act now to protect the planet."
result, n = perturb_length_extension(test_text, n_fillers=3)
print("P6 Length Extension test:")
print(f"Original words : {len(test_text.split())}")
print(f"Perturbed words: {len(result.split())}")
print(f"Fillers added  : {n}")
print(f"Result: {result}")
print()
print("P6 provides counterfactual: if score rises with filler, length drives scoring, not argument quality")

P6 Length Extension test:
Original words : 14
Perturbed words: 50
Fillers added  : 3
Result: Climate change is a serious issue. We must act now to protect the planet. It is therefore clear, based on the points raised above, that this is significant. As mentioned previously, this aspect of the argument deserves attention. This further reinforces the argument that has been made throughout this essay.

P6 provides counterfactual: if score rises with filler, length drives scoring, not argument quality


In [48]:
# ── COVERAGE CHECK: ALL 6 PERTURBATIONS ──────────────────────────────────
# Quick coverage check on 50 sample essays for P4, P5, P6

# Load discourse-level data for P4 and P5
df_disc = pd.read_csv(f'{DATA_PATH}/persuade_corpus_2.0_train.csv')

p4_hits, p5_hits, p6_hits = 0, 0, 0
sample_ids = df['essay_id'].iloc[:50].tolist()

for eid in sample_ids:
    _, ok4 = perturb_discourse_weakening(eid, df_disc)
    _, ok5 = perturb_structural_disruption(eid, df_disc)
    _, _   = perturb_length_extension(df[df['essay_id']==eid]['full_text'].values[0])
    if ok4: p4_hits += 1
    if ok5: p5_hits += 1
    p6_hits += 1  # Always applies

print("=" * 55)
print("COVERAGE CHECK — All 6 Perturbations (50 essays)")
print("=" * 55)
print(f"P1 Lexical Formality      : 98% (confirmed earlier)")
print(f"P2 Connective Density     : 100% (confirmed earlier)")
print(f"P3 Syntactic Voice        : 87% (confirmed earlier)")
print(f"P4 Discourse Weakening    : {p4_hits}/50 ({p4_hits*2}%)")
print(f"P5 Structural Disruption  : {p5_hits}/50 ({p5_hits*2}%)")
print(f"P6 Length Extension       : {p6_hits}/50 (100%)")
print()
print("P1-P3: Stylistic perturbations (surface level)")
print("P4-P5: Discourse-level perturbations (argument structure)")
print("P6: Length counterfactual (causal evidence)")
print()
print("Novel contribution: first study to compare stylistic vs")
print("discourse-level CIV using PERSUADE 2.0 annotations")

COVERAGE CHECK — All 6 Perturbations (50 essays)
P1 Lexical Formality      : 98% (confirmed earlier)
P2 Connective Density     : 100% (confirmed earlier)
P3 Syntactic Voice        : 87% (confirmed earlier)
P4 Discourse Weakening    : 11/50 (22%)
P5 Structural Disruption  : 10/50 (20%)
P6 Length Extension       : 50/50 (100%)

P1-P3: Stylistic perturbations (surface level)
P4-P5: Discourse-level perturbations (argument structure)
P6: Length counterfactual (causal evidence)

Novel contribution: first study to compare stylistic vs
discourse-level CIV using PERSUADE 2.0 annotations


## Correcting the Data Structure

The training CSV has 173,266 rows because each row is one **discourse segment** (Lead, Position, Claim, Evidence, Counterclaim, Rebuttal, Concluding Statement), not one essay.

To run perturbations correctly, we need:
1. **df_essays** — one row per unique essay (approximately 15,593 rows) for P1, P2, P3, P6
2. **df_disc** — the full 173,266 rows with discourse annotations for P4 and P5

The `discourse_effectiveness` column contains: Effective, Adequate, Ineffective
The `discourse_type` column contains: Lead, Position, Claim, Evidence, Counterclaim, Rebuttal, Concluding Statement

In [49]:
# SPLIT DATA INTO ESSAYS AND DISCOURSE SEGMENTS


# Load full training data
df_full = pd.read_csv('/kaggle/input/datasets/shravaniuk/persuade-corpus-2-0-tt/persuade_corpus_2.0_train.csv')
print(f"Full dataset rows (discourse segments): {len(df_full):,}")
print(f"Columns: {df_full.columns.tolist()}")

# df_disc = full discourse-level data for P4 and P5
df_disc = df_full.copy()
print(f"\nDiscourse data (df_disc): {len(df_disc):,} segments")
print(f"Discourse types: {df_disc['discourse_type'].unique().tolist()}")
print(f"Effectiveness labels: {df_disc['discourse_effectiveness'].unique().tolist()}")

# df_essays = one row per unique essay for P1, P2, P3, P6
df_essays = df_full.drop_duplicates(subset='essay_id_comp').reset_index(drop=True)
print(f"\nUnique essays (df_essays): {len(df_essays):,}")
print(f"Score distribution:")
print(df_essays['holistic_essay_score'].value_counts().sort_index())

# Verify the discourse data looks correct
print(f"\nSample discourse segments for first essay:")
first_essay_id = df_essays['essay_id_comp'].iloc[0]
sample_disc = df_disc[df_disc['essay_id_comp'] == first_essay_id][
    ['discourse_type', 'discourse_effectiveness', 'discourse_text']
]
for _, seg in sample_disc.iterrows():
    print(f"  [{seg['discourse_effectiveness']:12}] {seg['discourse_type']:25} | {str(seg['discourse_text'])[:80]}...")

Full dataset rows (discourse segments): 173,266
Columns: ['essay_id', 'essay_id_comp', 'competition_set', 'full_text', 'holistic_essay_score', 'discourse_id', 'discourse_start', 'discourse_end', 'discourse_text', 'discourse_type', 'discourse_type_num', 'discourse_effectiveness', 'hierarchical_id', 'hierarchical_text', 'hierarchical_label', 'provider', 'task', 'source_text', 'prompt_name', 'assignment', 'gender', 'grade_level', 'ell_status', 'race_ethnicity', 'economically_disadvantaged', 'student_disability_status', 'essay_word_count']

Discourse data (df_disc): 173,266 segments
Discourse types: ['Unannotated', 'Lead', 'Position', 'Evidence', 'Claim', 'Concluding Statement', 'Counterclaim', 'Rebuttal']
Effectiveness labels: [nan, 'Adequate', 'Ineffective', 'Effective']

Unique essays (df_essays): 15,594
Score distribution:
holistic_essay_score
1     610
2    3218
3    4943
4    4194
5    2082
6     547
Name: count, dtype: int64

Sample discourse segments for first essay:
  [         na

In [50]:
# RUN P1, P2, P3, P6 ON UNIQUE ESSAYS (NOT DISCOURSE SEGMENTS)

import time

results = []
failed = []

print(f"Starting perturbation run on {len(df_essays):,} unique essays...")
print("=" * 60)
start_time = time.time()

for idx, row in df_essays.iterrows():
    if idx % 1000 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"  Progress: {idx:,}/{len(df_essays):,} essays | {elapsed:.1f} mins elapsed")
    
    try:
        essay_id = row['essay_id_comp']
        original = str(row['full_text'])
        score    = row['holistic_essay_score']
        
        if pd.isna(row['full_text']) or len(original.strip()) < 10:
            failed.append({'essay_id': essay_id, 'error': 'Empty or too short'})
            continue
        
        # P1 Lexical Formality
        p1_text, p1_subs = perturb_lexical_formality(original)
        
        # P2 Connective Density
        p2_text, p2_ins = perturb_connective_density(original)
        
        # P3 Syntactic Voice
        p3_text, p3_conv = perturb_syntactic_voice(original, nlp)
        
        # P6 Length Extension
        p6_text, p6_n = perturb_length_extension(original, n_fillers=3)
        
        results.append({
            'essay_id': essay_id,
            'score': score,
            'original': original,
            'p1_formality': p1_text,   'p1_subs': p1_subs,
            'p2_density': p2_text,     'p2_insertions': p2_ins,
            'p3_voice': p3_text,       'p3_conversions': p3_conv,
            'p6_length': p6_text,      'p6_fillers': p6_n,
        })
        
    except Exception as e:
        failed.append({'essay_id': row.get('essay_id_comp', idx), 'error': str(e)})
        if len(failed) <= 5:
            print(f"\n  ERROR on essay {row.get('essay_id_comp', idx)}: {str(e)}")
        continue

elapsed_total = (time.time() - start_time) / 60
print("\n" + "=" * 60)
print(f"COMPLETED")
print(f"  Successful: {len(results):,} essays")
print(f"  Failed    : {len(failed):,} essays")
print(f"  Time      : {elapsed_total:.1f} minutes")

Starting perturbation run on 15,594 unique essays...
  Progress: 0/15,594 essays | 0.0 mins elapsed
  Progress: 1,000/15,594 essays | 2.4 mins elapsed
  Progress: 2,000/15,594 essays | 5.0 mins elapsed
  Progress: 3,000/15,594 essays | 7.4 mins elapsed
  Progress: 4,000/15,594 essays | 9.8 mins elapsed
  Progress: 5,000/15,594 essays | 11.7 mins elapsed
  Progress: 6,000/15,594 essays | 13.5 mins elapsed
  Progress: 7,000/15,594 essays | 15.4 mins elapsed
  Progress: 8,000/15,594 essays | 17.0 mins elapsed
  Progress: 9,000/15,594 essays | 18.9 mins elapsed
  Progress: 10,000/15,594 essays | 21.1 mins elapsed
  Progress: 11,000/15,594 essays | 23.2 mins elapsed
  Progress: 12,000/15,594 essays | 25.2 mins elapsed
  Progress: 13,000/15,594 essays | 27.3 mins elapsed
  Progress: 14,000/15,594 essays | 30.3 mins elapsed
  Progress: 15,000/15,594 essays | 33.0 mins elapsed

COMPLETED
  Successful: 15,594 essays
  Failed    : 0 essays
  Time      : 34.6 minutes


In [51]:
# FIX: Check which essay ID column P4/P5 functions use

# Show what the two ID columns look like
print("essay_id values (first 5):")
print(df_disc['essay_id'].head().tolist())
print(f"\nessay_id_comp values (first 5):")
print(df_disc['essay_id_comp'].head().tolist())

# Check which one the P4 function uses
# Test with both ID types for the first essay
test_essay_id_comp = df_essays['essay_id_comp'].iloc[0]
test_essay_id      = df_essays['essay_id'].iloc[0]

match_comp = df_disc[df_disc['essay_id_comp'] == test_essay_id_comp]
match_id   = df_disc[df_disc['essay_id'] == test_essay_id]

print(f"\nTest essay_id_comp = {test_essay_id_comp}")
print(f"  Matches in df_disc using essay_id_comp: {len(match_comp)}")

print(f"\nTest essay_id = {test_essay_id}")
print(f"  Matches in df_disc using essay_id: {len(match_id)}")

# Show discourse segments for this essay
if len(match_comp) > 0:
    print(f"\nDiscourse segments for this essay:")
    for _, seg in match_comp.iterrows():
        eff = seg['discourse_effectiveness'] if pd.notna(seg['discourse_effectiveness']) else 'Unannotated'
        print(f"  [{eff:12}] {str(seg['discourse_type']):25} | {str(seg['discourse_text'])[:80]}...")

essay_id values (first 5):
['5.40889E+12', '5.40889E+12', '5.40889E+12', '5.40889E+12', '5.40889E+12']

essay_id_comp values (first 5):
['423A1CA112E2', '423A1CA112E2', '423A1CA112E2', '423A1CA112E2', '423A1CA112E2']

Test essay_id_comp = 423A1CA112E2
  Matches in df_disc using essay_id_comp: 11

Test essay_id = 5.40889E+12
  Matches in df_disc using essay_id: 11

Discourse segments for this essay:
  [Unannotated ] Unannotated               | Phones

...
  [Adequate    ] Lead                      | Modern humans today are always on their phone. They are always on their phone mo...
  [Adequate    ] Position                  | They are some really bad consequences when stuff happens when it comes to a phon...
  [Adequate    ] Evidence                  | Some certain areas in the United States ban phones from class rooms just because...
  [Adequate    ] Evidence                  | When people have phones, they know about certain apps that they have .Apps like ...
  [Adequate    ] Claim   

In [52]:
# RUN P4 AND P5 — FIXED WITH CORRECT ID COLUMN

# First, check which ID column works
test_id_comp = df_essays['essay_id_comp'].iloc[0]
test_id      = df_essays['essay_id'].iloc[0]

if len(df_disc[df_disc['essay_id_comp'] == test_id_comp]) > 0:
    ID_COL = 'essay_id_comp'
elif len(df_disc[df_disc['essay_id'] == test_id]) > 0:
    ID_COL = 'essay_id'
else:
    print("ERROR: Cannot match essays to discourse segments!")
    ID_COL = None

print(f"Using ID column: {ID_COL}")

# Filter out Unannotated segments — they have no effectiveness label
df_disc_clean = df_disc[df_disc['discourse_effectiveness'].notna()].copy()
print(f"Discourse segments with effectiveness labels: {len(df_disc_clean):,}")
print(f"Effectiveness distribution:")
print(df_disc_clean['discourse_effectiveness'].value_counts())

print(f"\nRunning P4 and P5 on {len(df_essays):,} essays...")
print("=" * 60)

p4_results = []
p5_results = []
p4_success = 0
p5_success = 0

for idx, row in df_essays.iterrows():
    if idx % 2000 == 0:
        print(f"  Progress: {idx:,}/{len(df_essays):,} | P4 hits: {p4_success} | P5 hits: {p5_success}")
    
    essay_id = row[ID_COL]
    
    # Get discourse segments for this essay
    essay_segments = df_disc_clean[df_disc_clean[ID_COL] == essay_id]
    
    # ── P4: DISCOURSE WEAKENING ──
    # Find an Effective segment, replace with Ineffective of same type
    p4_text = None
    p4_ok = False
    try:
        effective_segs = essay_segments[essay_segments['discourse_effectiveness'] == 'Effective']
        
        if len(effective_segs) > 0:
            # Pick the first Effective segment
            target_seg = effective_segs.iloc[0]
            target_type = target_seg['discourse_type']
            target_text = str(target_seg['discourse_text'])
            
            # Find an Ineffective segment of the same type from ANY other essay
            replacement_pool = df_disc_clean[
                (df_disc_clean['discourse_effectiveness'] == 'Ineffective') &
                (df_disc_clean['discourse_type'] == target_type) &
                (df_disc_clean[ID_COL] != essay_id)
            ]
            
            if len(replacement_pool) > 0:
                # Pick a random replacement
                replacement = replacement_pool.sample(1, random_state=42).iloc[0]
                replacement_text = str(replacement['discourse_text'])
                
                # Replace in the full essay text
                original_text = str(row['full_text'])
                p4_text = original_text.replace(target_text, replacement_text, 1)
                
                if p4_text != original_text:
                    p4_ok = True
                    p4_success += 1
    except Exception as e:
        pass
    
    p4_results.append({'essay_id': row['essay_id_comp'], 'p4_weakening': p4_text, 'p4_applied': p4_ok})
    
    # ── P5: STRUCTURAL DISRUPTION ──
    # Remove Counterclaim or Rebuttal segment
    p5_text = None
    p5_ok = False
    try:
        counter_segs = essay_segments[
            essay_segments['discourse_type'].isin(['Counterclaim', 'Rebuttal'])
        ]
        
        if len(counter_segs) > 0:
            # Remove the first Counterclaim or Rebuttal found
            remove_seg = counter_segs.iloc[0]
            remove_text = str(remove_seg['discourse_text'])
            
            original_text = str(row['full_text'])
            p5_text = original_text.replace(remove_text, '', 1).strip()
            
            # Clean up double spaces and double newlines
            p5_text = re.sub(r'\n\s*\n\s*\n', '\n\n', p5_text)
            p5_text = re.sub(r'  +', ' ', p5_text)
            
            if p5_text != original_text:
                p5_ok = True
                p5_success += 1
    except Exception as e:
        pass
    
    p5_results.append({'essay_id': row['essay_id_comp'], 'p5_disruption': p5_text, 'p5_applied': p5_ok})

df_p4 = pd.DataFrame(p4_results)
df_p5 = pd.DataFrame(p5_results)

print(f"\n{'='*60}")
print(f"P4 Discourse Weakening  : {p4_success:,}/{len(df_essays):,} ({p4_success/len(df_essays)*100:.1f}%)")
print(f"P5 Structural Disruption: {p5_success:,}/{len(df_essays):,} ({p5_success/len(df_essays)*100:.1f}%)")

Using ID column: essay_id_comp
Discourse segments with effectiveness labels: 144,289
Effectiveness distribution:
discourse_effectiveness
Adequate       113244
Effective       24583
Ineffective      6462
Name: count, dtype: int64

Running P4 and P5 on 15,594 essays...
  Progress: 0/15,594 | P4 hits: 0 | P5 hits: 0
  Progress: 2,000/15,594 | P4 hits: 446 | P5 hits: 490
  Progress: 4,000/15,594 | P4 hits: 1011 | P5 hits: 941
  Progress: 6,000/15,594 | P4 hits: 1268 | P5 hits: 1212
  Progress: 8,000/15,594 | P4 hits: 1661 | P5 hits: 1760
  Progress: 10,000/15,594 | P4 hits: 2115 | P5 hits: 2239
  Progress: 12,000/15,594 | P4 hits: 2639 | P5 hits: 3039
  Progress: 14,000/15,594 | P4 hits: 3094 | P5 hits: 3829

P4 Discourse Weakening  : 3,533/15,594 (22.7%)
P5 Structural Disruption: 3,947/15,594 (25.3%)


In [53]:
# MERGE ALL 6 PERTURBATIONS INTO ONE DATASET

df_perturbed = pd.DataFrame(results)

# Merge P4 and P5 results
df_perturbed = df_perturbed.merge(df_p4, on='essay_id', how='left')
df_perturbed = df_perturbed.merge(df_p5, on='essay_id', how='left')

# Save
df_perturbed.to_csv('/kaggle/working/perturbations_full_all6.csv', index=False)

print(f"Saved: perturbations_full_all6.csv")
print(f"Shape: {df_perturbed.shape}")
print(f"Columns: {df_perturbed.columns.tolist()}")

print(f"\nFinal coverage — ALL 6 PERTURBATIONS:")
print(f"  P1 Lexical Formality  : {(df_perturbed['p1_subs'] > 0).sum():,}/{len(df_perturbed):,}")
print(f"  P2 Connective Density : {(df_perturbed['p2_insertions'] > 0).sum():,}/{len(df_perturbed):,}")
print(f"  P3 Syntactic Voice    : {(df_perturbed['p3_conversions'] > 0).sum():,}/{len(df_perturbed):,}")
print(f"  P4 Discourse Weakening: {df_perturbed['p4_applied'].sum():,}/{len(df_perturbed):,}")
print(f"  P5 Structural Disrupt : {df_perturbed['p5_applied'].sum():,}/{len(df_perturbed):,}")
print(f"  P6 Length Extension   : {(df_perturbed['p6_fillers'] > 0).sum():,}/{len(df_perturbed):,}")

print(f"\nPhase 3 Step 1 — ALL 6 PERTURBATIONS COMPLETE")
print(f"Next step: SBERT semantic gate (new notebook)")

Saved: perturbations_full_all6.csv
Shape: (15594, 15)
Columns: ['essay_id', 'score', 'original', 'p1_formality', 'p1_subs', 'p2_density', 'p2_insertions', 'p3_voice', 'p3_conversions', 'p6_length', 'p6_fillers', 'p4_weakening', 'p4_applied', 'p5_disruption', 'p5_applied']

Final coverage — ALL 6 PERTURBATIONS:
  P1 Lexical Formality  : 15,368/15,594
  P2 Connective Density : 15,570/15,594
  P3 Syntactic Voice    : 13,443/15,594
  P4 Discourse Weakening: 3,533/15,594
  P5 Structural Disrupt : 3,947/15,594
  P6 Length Extension   : 15,594/15,594

Phase 3 Step 1 — ALL 6 PERTURBATIONS COMPLETE
Next step: SBERT semantic gate (new notebook)
